# **nn.Module로 XOR 리팩토링하기**

## **1. 데이터 준비**

**Q1. XOR을 학습시킬 데이터를 생성합니다.**

In [1]:
import torch

X = [[0, 0], [0, 1], [1, 1], [1, 0]]
y = [0, 1, 0, 1]

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).view(4, 1)

print(X_tensor.shape, y_tensor.shape)

# MLP 출력 예상: [0, 0, 1, 1] -> shape: (4, 1)
#   - 예상 출력: shape와 맞춰주지 않으면 버그!

torch.Size([4, 2]) torch.Size([4, 1])


## **2. 신경망 생성하기**

**Q1. MLP 클래스를 만든다.**

- __init__() : Linear Layer 2개와 sigmoid 레이어 1개를 생성한다.
  - fc1 : 2개의 input과 2개의 output을 가진 fully connected layer
  - fc2 : 2개의 input과 1개의 output을 가진 fully connected layer
  - sigmoid : sigmoid 활성화 함수 layer
- forward() :
  - fc1 layer -> sigmoid -> fc2 layer -> sigmoid

In [2]:
from torch import nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(2, 2) # requires_grad = True (가중치가 미분 대상이라고 자동 등록)
        self.fc2 = nn.Linear(2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, X):
        z1 = self.fc1(X)
        a1 = self.sigmoid(z1)
        z2 = self.fc2(a1)
        a2 = self.sigmoid(z2)

        return a2

### Pytorch를 활용한 일반적인 코드 구조

1. 모델 구조와 forward는 하나의 class로 분리
2. 손실 계산, 역전파, 갱신 --> 외부로 분리

## **3. Loss와 Gradient Descent**

**Q2. Loss 인스턴스를 만든다.**

criterion이라는 이름으로 nn.BCELoss의 인스턴스를 생성한다.

In [3]:
criterion = nn.BCELoss()

**Q3. Gradient Descent를 구현한다.**

- torch.no_grad() : 그래프에 기록되는것을 방지한다.
- model.parameters() : weight, bias등의 가중치를 반환한다.

In [4]:
def gd(model, lr):
    with torch.no_grad():
        for params in model.parameters():
            params -= lr * params.grad

## **4. 모델 학습**

**Q3. 모델을 학습시킨다.**
- learning rate : 1.0
- epoch : 1000

forward -> loss -> backward -> gradient descent 순으로 1개의 epoch를 완성한다.
- 한번의 epoch 이후 다음 epoch 시작 전, gradient 초기화를 해줘야 한다.

100번의 epoch마다 loss를 출력하게 한다.

model은 train모드로 사용해야 한다.

In [5]:
model = MLP()

In [ ]:
lr = 1.0
epochs = 1000

for epoch in range(epochs):
    model.train() # train mode 설정 -> parameter를 수정하겠다 알림
    y_pred = model(X_tensor) # 1. forward
    loss = criterion(y_pred, y_tensor) # 2. loss calculation
    loss.backward() # 3. back propagation
    gd(model, lr) # 4. Gradient Descent
    # 이전의 gradient 정보가 남아있음
    model.zero_grad() # gradient 초기화

    if epoch % 100 == 0:
        print(f"epoch {epoch} | loss {loss}")

epoch 0 | loss 0.7090588808059692
epoch 100 | loss 0.6931675672531128
epoch 200 | loss 0.6931111812591553
epoch 300 | loss 0.6930325627326965
epoch 400 | loss 0.6928524374961853
epoch 500 | loss 0.6922560930252075
epoch 600 | loss 0.688569962978363
epoch 700 | loss 0.6459229588508606
epoch 800 | loss 0.4839613735675812
epoch 900 | loss 0.40442028641700745


## **5. 결과**

**Q1. 모델을 eval 모드로, 예측 결과를 확인해보자.**

In [7]:
model.eval()

with torch.no_grad():
    y_pred = model(X_tensor)
    print(y_pred)

tensor([[0.0370],
        [0.4906],
        [0.5155],
        [0.9567]])
